<a href="https://colab.research.google.com/github/MirMurtazaa/agent-miscon/blob/main/minimax_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import io

uploaded = files.upload()

dfs = []
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  # Read the CSV into a pandas DataFrame
  df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
  dfs.append(df)

# You can now access the dataframes in the 'dfs' list
# For example, to display the first dataframe:
# display(dfs[0].head())

Saving e4_evaluation_results.csv to e4_evaluation_results.csv
Saving e4_experiment_summary.csv to e4_experiment_summary.csv
Saving e4_evaluation_strategy_log.csv to e4_evaluation_strategy_log.csv
Saving e4_evaluation_trace_log.csv to e4_evaluation_trace_log.csv
Saving e4_training_data.csv to e4_training_data.csv
User uploaded file "e4_evaluation_results.csv" with length 11106 bytes
User uploaded file "e4_experiment_summary.csv" with length 280 bytes
User uploaded file "e4_evaluation_strategy_log.csv" with length 145108 bytes
User uploaded file "e4_evaluation_trace_log.csv" with length 106537 bytes
User uploaded file "e4_training_data.csv" with length 120720 bytes


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal, spearmanr
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import warnings
import os
import importlib.util
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')

# ============================================================================
# SCRIPT CONFIGURATION
# ============================================================================

OUTPUT_FOLDER = 'analysis_figures'

META_STRATEGY_NAMES = [
    'HUB_TARGETING', 'BRIDGE_TARGETING', 'CLOSENESS_CENTRALITY',
    'K_CORE_TARGETING', 'PERIPHERAL_TARGETING', 'HIGH_CLUSTERING',
    'OPPONENT_NEIGHBOR'
]

EMPIRICAL_DATASETS = ['POLBLOGS', 'FACEBOOK', 'EMAIL']

EXPERIMENTS_CONFIG = [
    ('e1', 'EXP1', 'IND vs. IND (EXP1)', "Indep_vs_Indep", 'independent', 'independent'),
    ('e2', 'EXP2', 'MM vs. IND (EXP2)', "Minimax_vs_Indep", 'minimax', 'independent'),
    ('e3', 'EXP3', 'MM vs. MM (EXP3)', "Minimax_vs_Minimax", 'minimax', 'minimax'),
    ('e4', 'EXP4', 'IND vs. MM (EXP4)', "Indep_vs_Minimax", 'independent', 'minimax')
]

EXPERIMENT_NAMES = {cfg[0]: f"{cfg[1]}: {cfg[3]} ({cfg[4]} vs {cfg[5]})" for cfg in EXPERIMENTS_CONFIG}
EXPERIMENT_AGENT_TYPES = {
    cfg[0]: {"display_name": cfg[3], "short_name": cfg[1], "legend_label": cfg[2],
             "blocker_type": cfg[4], "adversary_type": cfg[5]}
    for cfg in EXPERIMENTS_CONFIG
}
EXPERIMENT_LEGEND_LABELS = {cfg[0]: cfg[2] for cfg in EXPERIMENTS_CONFIG}
EXPERIMENT_SHORT_NAMES = {cfg[0]: cfg[1] for cfg in EXPERIMENTS_CONFIG}

SCIENTIFIC_PALETTE = [
    '#0072B2', '#D55E00', '#009E73', '#E69F00',
    '#CC79A7', '#56B4E9', '#A0522D', '#F0E442', '#999999'
]

STRATEGY_COLOR_MAP = dict(zip(META_STRATEGY_NAMES, SCIENTIFIC_PALETTE))
NAN_COLOR = '#e0e0e0'

plt.style.use('seaborn-v0_8-ticks')
sns.set_palette(sns.color_palette(SCIENTIFIC_PALETTE))
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '--'

KALEIDO_INSTALLED = importlib.util.find_spec("kaleido") is not None

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def map_strategy_id_to_name(strategy_id):
    try:
        return META_STRATEGY_NAMES[int(strategy_id)]
    except (IndexError, ValueError, TypeError):
        return f"Unknown_ID_{strategy_id}"

def hex_to_rgba(hex_color, alpha):
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

def extract_network_type(network_name):
    for emp_dataset in EMPIRICAL_DATASETS:
        if emp_dataset in network_name.upper():
            return emp_dataset
    if '-' in network_name:
        return network_name.split('-')[0]
    return network_name

def load_all_experiments():
    experiments = {}
    for file_prefix, _, _, display_name, _, _ in EXPERIMENTS_CONFIG:
        exp_id = file_prefix
        try:
            exp_results_file = f'exp_{exp_id}_evaluation_results.csv'
            if not os.path.exists(exp_results_file):
                exp_results_file = f'{exp_id}_evaluation_results.csv'

            if os.path.exists(exp_results_file):
                experiments[exp_id] = {
                    'results': pd.read_csv(exp_results_file),
                    'strategy': pd.read_csv(f'exp_{exp_id}_evaluation_strategy_log.csv') if os.path.exists(f'exp_{exp_id}_evaluation_strategy_log.csv') else pd.read_csv(f'{exp_id}_evaluation_strategy_log.csv'),
                    'trace': pd.read_csv(f'exp_{exp_id}_evaluation_trace_log.csv') if os.path.exists(f'exp_{exp_id}_evaluation_trace_log.csv') else pd.read_csv(f'{exp_id}_evaluation_trace_log.csv'),
                    'training': pd.read_csv(f'exp_{exp_id}_training_data.csv') if os.path.exists(f'exp_{exp_id}_training_data.csv') else (pd.read_csv(f'{exp_id}_training_data.csv') if os.path.exists(f'{exp_id}_training_data.csv') else None)
                }
                print(f"Loaded experiment data for: {exp_id} (as {display_name})")
            else:
                print(f"Warning: Results file not found for experiment {exp_id}")
        except Exception as e:
            print(f"Error loading experiment {exp_id}: {e}")
    return experiments

# ============================================================================
# 1. OVERALL PERFORMANCE (BOXPLOT)
# ============================================================================

def plot_fig1a_influence_boxplot(df_filtered, phase_name):
    print(f"  Generating Figure 1a: Influence Boxplot ({phase_name})")
    plt.figure(figsize=(12, 8))
    hue_order = [cfg[2] for cfg in EXPERIMENTS_CONFIG]

    ax = sns.boxplot(data=df_filtered, x='network_type', y='norm_influence',
                     hue='legend_label', hue_order=hue_order)

    ax.set_xlabel('Network Type')
    ax.set_ylabel('Normalized Influence')
    ax.legend(title='Experiment Matchup', bbox_to_anchor=(1.05, 1), loc='upper left')

    n_groups = len(ax.get_xticks())
    for i in range(n_groups - 1):
        ax.axvline(i + 0.5, color='gray', linestyle=':', alpha=0.3)

    sns.despine(ax=ax, trim=True)
    plt.savefig(f'{OUTPUT_FOLDER}/fig_1a_influence_boxplot_{phase_name}.png', bbox_inches='tight')
    plt.close()

def plot_fig1b_generalization_barplot(df_filtered, phase_name):
    print(f"  Generating Figure 1b: Generalization Barplot ({phase_name})")
    if phase_name == 'COMBINED':
        plt.figure(figsize=(10, 6))
        hue_order = [cfg[2] for cfg in EXPERIMENTS_CONFIG]
        phase_data = df_filtered.groupby(['legend_label', 'phase'])['final_blocker_reward'].mean().reset_index()
        phase_data['phase'] = pd.Categorical(phase_data['phase'], categories=['train', 'test'], ordered=True)
        phase_data = phase_data.sort_values('phase')
        ax = sns.barplot(data=phase_data, x='phase', y='final_blocker_reward',
                         hue='legend_label', hue_order=hue_order)
        ax.set_xlabel('Phase')
        ax.set_ylabel('Mean Blocker Reward')
        ax.legend(title='Experiment Matchup', bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        synthetic_data = df_filtered[~df_filtered['network_type'].isin(EMPIRICAL_DATASETS)]
        empirical_data = df_filtered[df_filtered['network_type'].isin(EMPIRICAL_DATASETS)]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        if not synthetic_data.empty:
            syn_data = synthetic_data.groupby(['legend_label'])['final_blocker_reward'].mean().reset_index()
            hue_order = [cfg[2] for cfg in EXPERIMENTS_CONFIG]
            sns.barplot(data=syn_data, x='legend_label', y='final_blocker_reward',
                       order=hue_order, palette=sns.color_palette(SCIENTIFIC_PALETTE, n_colors=len(hue_order)), ax=ax1)
            ax1.set_title(f'Synthetic Networks ({phase_name})')
            ax1.set_xlabel('Experiment Matchup'); ax1.set_ylabel('Mean Blocker Reward')
            ax1.tick_params(axis='x', rotation=15)

        if not empirical_data.empty:
            emp_data = empirical_data.groupby(['legend_label'])['final_blocker_reward'].mean().reset_index()
            hue_order = [cfg[2] for cfg in EXPERIMENTS_CONFIG]
            sns.barplot(data=emp_data, x='legend_label', y='final_blocker_reward',
                       order=hue_order, palette=sns.color_palette(SCIENTIFIC_PALETTE, n_colors=len(hue_order)), ax=ax2)
            ax2.set_title(f'Empirical Networks ({phase_name})')
            ax2.set_xlabel('Experiment Matchup'); ax2.set_ylabel('Mean Blocker Reward')
            ax2.tick_params(axis='x', rotation=15)
        plt.tight_layout()

    sns.despine(trim=True)
    plt.savefig(f'{OUTPUT_FOLDER}/fig_1b_generalization_barplot_{phase_name}.png', bbox_inches='tight')
    plt.close()

# ============================================================================
# 2. NETWORK PROPERTY ANALYSIS (REGPLOT)
# ============================================================================

def plot_fig2_regplot(df_filtered, prop_name, x_label, phase_name, experiment_labels, marker_map):
    print(f"  Generating Figure 2: Regression Plot for {prop_name} ({phase_name})")
    if prop_name not in df_filtered.columns: return

    plt.figure(figsize=(12, 8))
    ax = plt.gca()

    synthetic_data = df_filtered[~df_filtered['network_type'].isin(EMPIRICAL_DATASETS)]
    empirical_data = df_filtered[df_filtered['network_type'].isin(EMPIRICAL_DATASETS)]

    palette = sns.color_palette(SCIENTIFIC_PALETTE, n_colors=len(experiment_labels))
    colors = {label: color for label, color in zip(experiment_labels, palette)}

    STANDARD_SIZE = 70
    LEGEND_MARKER_SIZE = 9

    synth_colors = {
        'BA': '#5D3A9B',  # Deep Purple
        'ER': '#E66100',  # Distinct Orange
        'WS': '#117733'   # Green
    }

    # 1. Plot Synthetic
    if not synthetic_data.empty:
        for net_type in sorted(synthetic_data['network_type'].unique()):
            subset = synthetic_data[synthetic_data['network_type'] == net_type]
            marker = marker_map.get(net_type, 'o')
            c = synth_colors.get(net_type, 'gray')
            ax.scatter(subset[prop_name], subset['norm_influence'],
                       marker=marker, color=c, alpha=0.5,
                       s=STANDARD_SIZE, edgecolors='white', linewidth=0.5)

    # 2. Plot Empirical
    emp_palette = ['#FF6B6B', '#4ECDC4', '#FFD166']
    if not empirical_data.empty:
        unique_emp = sorted(empirical_data['network_type'].unique())
        for i, emp_name in enumerate(unique_emp):
            subset = empirical_data[empirical_data['network_type'] == emp_name]
            marker = marker_map.get(emp_name, 'D')
            color = emp_palette[i % len(emp_palette)]
            ax.scatter(subset[prop_name], subset['norm_influence'],
                      marker=marker, color=color, s=STANDARD_SIZE, alpha=0.9,
                      edgecolors='black', linewidth=1)

    # 3. Regression Lines
    for label in experiment_labels:
        exp_data = df_filtered[df_filtered['legend_label'] == label]
        if not exp_data.empty:
            sns.regplot(data=exp_data, x=prop_name, y='norm_influence',
                        ax=ax, color=colors[label], label=label,
                        scatter=False, ci=95, truncate=False)

    # 4. Legend
    custom_lines = []
    custom_labels = []

    custom_lines.append(Line2D([0], [0], color='white', label='--- Experiments ---'))
    custom_labels.append('--- Experiments ---')
    for label in experiment_labels:
        custom_lines.append(Line2D([0], [0], color=colors[label], lw=2))
        custom_labels.append(label)

    custom_lines.append(Line2D([0], [0], color='white', label=''))
    custom_labels.append('')
    custom_lines.append(Line2D([0], [0], color='white', label='--- Network Types ---'))
    custom_labels.append('--- Network Types ---')

    all_types = sorted(df_filtered['network_type'].unique())
    unique_emp_sorted = sorted(list(set(EMPIRICAL_DATASETS) & set(all_types)))

    for net_type in all_types:
        marker = marker_map.get(net_type, 'o')
        if net_type in EMPIRICAL_DATASETS:
            try:
                idx = unique_emp_sorted.index(net_type)
                c = emp_palette[idx % len(emp_palette)]
                edge_c = 'black'
            except: c='red'; edge_c='black'
        else:
            c = synth_colors.get(net_type, 'gray')
            edge_c = 'white'

        custom_lines.append(Line2D([0], [0], marker=marker, color='w',
                                  markerfacecolor=c, markersize=LEGEND_MARKER_SIZE,
                                  markeredgecolor=edge_c, markeredgewidth=0.5))
        custom_labels.append(net_type)

    ax.legend(custom_lines, custom_labels, bbox_to_anchor=(1.05, 1), loc='upper left')

    ax.set_xlabel(x_label)
    ax.set_ylabel('Normalized Influence')
    sns.despine(ax=ax, trim=True)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_FOLDER}/fig_2_regplot_{prop_name}_{phase_name}.png', bbox_inches='tight')
    plt.close()

# ============================================================================
# 3. INFLUENCE TRACES
# ============================================================================

def plot_fig3_consolidated_traces(combined_trace_df):
    if combined_trace_df.empty or 'network_type' not in combined_trace_df.columns: return

    train_color = SCIENTIFIC_PALETTE[0]
    test_color = '#D62728'

    network_types = sorted(combined_trace_df['network_type'].dropna().unique())
    experiment_labels = [cfg[2] for cfg in EXPERIMENTS_CONFIG]

    for net_type in network_types:
        print(f"  Generating Figure 3 (Consolidated) for Network Type: {net_type}")
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), squeeze=False, sharex=True, sharey=True)
        axes = axes.flatten()

        for idx, label in enumerate(experiment_labels):
            ax = axes[idx]
            exp_net_data = combined_trace_df[
                (combined_trace_df['legend_label'] == label) &
                (combined_trace_df['network_type'] == net_type)
            ]

            if exp_net_data.empty:
                ax.set_title(f"{label}\n(No data)", fontsize=14)
                sns.despine(ax=ax, trim=True)
                continue

            for (game_num, phase), game_data in exp_net_data.groupby(['game_num', 'phase']):
                alpha = 0.1 if phase == 'train' else 0.25
                c = test_color if phase == 'test' else train_color
                ax.plot(game_data['round'], game_data['norm_influence'],
                        color=c, alpha=alpha, linewidth=1)

            mean_trace_train = exp_net_data[exp_net_data['phase']=='train'].groupby('round')['norm_influence'].mean()
            mean_trace_test = exp_net_data[exp_net_data['phase']=='test'].groupby('round')['norm_influence'].mean()

            if not mean_trace_train.empty:
                ax.plot(mean_trace_train.index, mean_trace_train.values,
                        color=train_color, linewidth=3, label='Mean (Train)', linestyle='-')
            if not mean_trace_test.empty:
                ax.plot(mean_trace_test.index, mean_trace_test.values,
                        color=test_color, linewidth=3, label='Mean (Test)', linestyle='--')

            title = f"{label} - {net_type}"
            if net_type in EMPIRICAL_DATASETS: title += " (Empirical)"
            ax.set_title(title, fontsize=14)
            ax.set_xlabel('Round')
            ax.set_ylabel('Normalized Influence')
            ax.legend(loc='best')
            sns.despine(ax=ax, trim=True)

        for idx in range(len(experiment_labels), len(axes)):
            fig.delaxes(axes[idx])

        plt.tight_layout()
        safe_net_type = net_type.replace('.', '_').replace('/', '_')
        plt.savefig(f'{OUTPUT_FOLDER}/fig_3_consolidated_{safe_net_type}_trace.png', bbox_inches='tight')
        plt.close(fig)

def plot_fig3_single_network_trace(network_trace_df, net_name):
    if network_trace_df.empty: return
    print(f"  Generating Figure 3 (Single) for Network: {net_name}")

    train_color = SCIENTIFIC_PALETTE[0]
    test_color = '#D62728'

    experiment_labels = [cfg[2] for cfg in EXPERIMENTS_CONFIG]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12), squeeze=False, sharex=True, sharey=True)
    axes = axes.flatten()

    for idx, label in enumerate(experiment_labels):
        ax = axes[idx]
        exp_net_data = network_trace_df[network_trace_df['legend_label'] == label]

        if exp_net_data.empty:
            ax.set_title(f"{label}\n(No data)", fontsize=14)
            sns.despine(ax=ax, trim=True)
            continue

        for (game_num, phase), game_data in exp_net_data.groupby(['game_num', 'phase']):
            alpha = 0.1 if phase == 'train' else 0.25
            c = test_color if phase == 'test' else train_color
            ax.plot(game_data['round'], game_data['norm_influence'],
                    color=c, alpha=alpha, linewidth=1)

        mean_trace_train = exp_net_data[exp_net_data['phase']=='train'].groupby('round')['norm_influence'].mean()
        mean_trace_test = exp_net_data[exp_net_data['phase']=='test'].groupby('round')['norm_influence'].mean()

        if not mean_trace_train.empty:
            ax.plot(mean_trace_train.index, mean_trace_train.values,
                    color=train_color, linewidth=3, label='Mean (Train)', linestyle='-')
        if not mean_trace_test.empty:
            ax.plot(mean_trace_test.index, mean_trace_test.values,
                    color=test_color, linewidth=3, label='Mean (Test)', linestyle='--')

        title = f"{label} - {net_name}"
        if extract_network_type(net_name) in EMPIRICAL_DATASETS: title += " (Empirical)"
        ax.set_title(title, fontsize=14)
        ax.set_xlabel('Round')
        ax.set_ylabel('Normalized Influence')
        ax.legend(loc='best')
        sns.despine(ax=ax, trim=True)

    for idx in range(len(experiment_labels), len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    safe_net_name = net_name.replace('.', '_').replace('/', '_')
    plt.savefig(f'{OUTPUT_FOLDER}/fig_3_single_{safe_net_name}_trace.png', bbox_inches='tight')
    plt.close(fig)

# ============================================================================
# 4. STRATEGY EVOLUTION - UPDATED WITH FIX
# ============================================================================

def plot_fig4_strategy_evolution(strategy_data, results_df, plot_title, filename):
    """
    Plots strategy evolution and pulls influence stats from results_df (ground truth).
    """
    if strategy_data.empty: return
    print(f"  Generating Figure 4: {plot_title}")

    # 1. Melt Strategy Data
    id_vars = ['network', 'network_type', 'legend_label', 'round', 'game_num',
               'experiment', 'blocker_type', 'adversary_type']

    strategy_melt = pd.melt(strategy_data,
                            id_vars=id_vars,
                            value_vars=['blocker_strategy', 'adversary_strategy'],
                            var_name='Agent', value_name='Strategy')
    strategy_melt['Agent'] = strategy_melt['Agent'].map({'blocker_strategy': 'Blocker', 'adversary_strategy': 'Adversary'})

    # 2. Setup Plot
    col_order = [cfg[1] for cfg in EXPERIMENTS_CONFIG]

    master_order = META_STRATEGY_NAMES
    present_strategies = set(strategy_melt['Strategy'].unique())
    known_strategies = set(META_STRATEGY_NAMES)
    unknown_strategies = sorted(list(present_strategies - known_strategies))
    final_hue_order = master_order + unknown_strategies
    current_plot_palette = STRATEGY_COLOR_MAP.copy()
    for unknown in unknown_strategies: current_plot_palette[unknown] = '#333333'

    g = sns.displot(
        data=strategy_melt, x="round", hue="Strategy",
        hue_order=final_hue_order, palette=current_plot_palette,
        col="experiment", row="Agent",
        col_order=col_order, row_order=['Blocker', 'Adversary'],
        multiple="fill", kind="hist", bins=10,
        height=3.5, aspect=1.3
    )

    g.set_axis_labels("Game Round (Binned)", "Strategy Proportion")
    g.set_titles("")

    # 3. Add Titles with Stats from RESULTS_DF (Not Strategy Data)
    agents = ['Blocker', 'Adversary']
    current_net_type = strategy_data['network_type'].iloc[0] # Assume homogeneous network type per plot call

    for row_idx, agent in enumerate(agents):
        for col_idx, exp in enumerate(col_order):
            ax = g.axes[row_idx, col_idx]

            # Lookup Stats in the MAIN results DF
            stats_subset = results_df[
                (results_df['experiment'] == exp) &
                (results_df['network_type'] == current_net_type)
            ]

            short_lbl = EXPERIMENT_LEGEND_LABELS.get(
                next((c[0] for c in EXPERIMENTS_CONFIG if c[1] == exp), ""), exp
            )

            stats_str = ""
            if not stats_subset.empty:
                val_mean = stats_subset['norm_influence'].mean()
                val_std = stats_subset['norm_influence'].std()
                if pd.notna(val_mean):
                    stats_str = f"\nMean Inf: {val_mean:.3f} ± {val_std:.3f}"
                else:
                    stats_str = "\nMean Inf: N/A"
            else:
                stats_str = "\nMean Inf: N/A (No Data)"

            # Check if this specific subplot has strategy data (to decide title vs empty)
            strat_subset = strategy_melt[
                (strategy_melt['Agent'] == agent) &
                (strategy_melt['experiment'] == exp)
            ]

            if not strat_subset.empty:
                ax.set_title(f"{agent}: {short_lbl}{stats_str}", fontsize=10)
            else:
                ax.set_title(f"{agent}: {exp} (No Data)", fontsize=10)

    plt.savefig(filename, bbox_inches='tight')
    plt.close()

# ============================================================================
# 5. STRATEGY FREQUENCY HEATMAP
# ============================================================================

def plot_fig5_strategy_frequency_heatmap(combined_strategy_df):
    if combined_strategy_df.empty: return
    print(f"  Generating Figure 5: Consolidated Strategy Frequency Heatmap")

    synthetic_data = combined_strategy_df[~combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]
    empirical_data = combined_strategy_df[combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    heatmap_cmap = 'YlOrBr'

    if not synthetic_data.empty:
        strategy_counts_synth = pd.crosstab(
            synthetic_data['blocker_strategy'], synthetic_data['adversary_strategy'],
            normalize=True
        ) * 100
        sns.heatmap(strategy_counts_synth, annot=True, fmt='.2f', cmap=heatmap_cmap,
                    cbar_kws={'label': 'Frequency (%)'}, ax=ax1)
        ax1.set_title('Synthetic Networks', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Adversary Strategy'); ax1.set_ylabel('Blocker Strategy')

    if not empirical_data.empty:
        strategy_counts_real = pd.crosstab(
            empirical_data['blocker_strategy'], empirical_data['adversary_strategy'],
            normalize=True
        ) * 100
        sns.heatmap(strategy_counts_real, annot=True, fmt='.2f', cmap=heatmap_cmap,
                    cbar_kws={'label': 'Frequency (%)'}, ax=ax2)
        ax2.set_title('Empirical Networks', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Adversary Strategy'); ax2.set_ylabel('Blocker Strategy')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_FOLDER}/fig_5_strategy_frequency_heatmap.png', bbox_inches='tight')
    plt.close()

# ============================================================================
# 6. PAYOFF MATRIX HEATMAP
# ============================================================================

def plot_fig6_payoff_matrix(combined_strategy_df):
    if combined_strategy_df.empty or 'norm_influence' not in combined_strategy_df.columns: return
    print(f"  Generating Figure 6: Payoff Matrix Heatmap")

    synthetic_data = combined_strategy_df[~combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]
    empirical_data = combined_strategy_df[combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    if not synthetic_data.empty:
        payoff_data = synthetic_data.groupby(['blocker_strategy', 'adversary_strategy'])['norm_influence'].mean()
        matrix = payoff_data.reset_index().pivot(index='blocker_strategy', columns='adversary_strategy', values='norm_influence')
        cmap = plt.get_cmap('viridis'); cmap.set_bad(color=NAN_COLOR)
        sns.heatmap(matrix, annot=True, fmt='.3f', cmap=cmap, cbar_kws={'label': 'Mean Norm Inf'}, linewidths=.5, ax=ax1)
        ax1.set_title('Synthetic Networks', fontsize=14, fontweight='bold')

    if not empirical_data.empty:
        payoff_data = empirical_data.groupby(['blocker_strategy', 'adversary_strategy'])['norm_influence'].mean()
        matrix = payoff_data.reset_index().pivot(index='blocker_strategy', columns='adversary_strategy', values='norm_influence')
        cmap = plt.get_cmap('plasma'); cmap.set_bad(color=NAN_COLOR)
        sns.heatmap(matrix, annot=True, fmt='.3f', cmap=cmap, cbar_kws={'label': 'Mean Norm Inf'}, linewidths=.5, ax=ax2)
        ax2.set_title('Empirical Networks', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_FOLDER}/fig_6_payoff_matrix_heatmap.png', bbox_inches='tight')
    plt.close()

# ============================================================================
# 7. FACETED FREQUENCY HEATMAPS
# ============================================================================

def plot_fig7_faceted_frequency_heatmaps(combined_strategy_df, all_blocker_strategies, all_adversary_strategies):
    if combined_strategy_df.empty or 'network_type' not in combined_strategy_df.columns: return
    print(f"  Generating Figure 7: Faceted Strategy Frequency Heatmaps")
    heatmap_cmap = 'YlOrBr'

    empirical_data = combined_strategy_df[combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]
    if not empirical_data.empty:
        _plot_fig7_subset(empirical_data, all_blocker_strategies, all_adversary_strategies, heatmap_cmap, 'empirical')

    synthetic_data = combined_strategy_df[~combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]
    if not synthetic_data.empty:
        _plot_fig7_subset(synthetic_data, all_blocker_strategies, all_adversary_strategies, heatmap_cmap, 'synthetic')

def _plot_fig7_subset(df_subset, all_b, all_a, cmap, suffix):
    group_data = df_subset.groupby(['network_type', 'legend_label']).apply(
        lambda x: pd.crosstab(x['blocker_strategy'], x['adversary_strategy'], normalize=True) * 100
    )
    if group_data.empty: return

    melted_data = group_data.reset_index().melt(
        id_vars=['network_type', 'legend_label', 'blocker_strategy'],
        var_name='adversary_strategy', value_name='frequency'
    )

    def draw_heatmap(*args, **kwargs):
        data = kwargs.pop('data')
        if data.empty: return
        heatmap_data = data.pivot(index='blocker_strategy', columns='adversary_strategy', values='frequency')
        heatmap_data = heatmap_data.reindex(index=all_b, columns=all_a).fillna(0)
        sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap=cmap, cbar=False,
                    vmin=0, vmax=melted_data['frequency'].max(), **kwargs)

    unique_nets = sorted(df_subset['network_type'].unique())
    unique_exps = [cfg[2] for cfg in EXPERIMENTS_CONFIG]

    g = sns.FacetGrid(
        melted_data, row='network_type', col='legend_label',
        col_order=unique_exps,
        row_order=unique_nets,
        margin_titles=True, height=4, aspect=1
    )
    g.map_dataframe(draw_heatmap)
    g.set_axis_labels('Adversary Strategy', 'Blocker Strategy')
    g.set_titles(col_template="{col_name}", row_template="{row_name}")
    for ax in g.axes.flat:
        for label in ax.get_xticklabels(): label.set_rotation(90)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_FOLDER}/fig_7_faceted_frequency_heatmaps_{suffix}.png', bbox_inches='tight')
    plt.close()

# ============================================================================
# 8. SANKEY DIAGRAMS
# ============================================================================

def plot_fig8_network_type_sankeys(combined_strategy_df):
    if combined_strategy_df.empty: return

    unique_types = sorted(combined_strategy_df['network_type'].unique())
    print(f"  Generating Figure 8: Individual Sankey Diagrams for: {', '.join(unique_types)}")
    pio.templates.default = "plotly_white"

    for net_type in unique_types:
        net_data = combined_strategy_df[combined_strategy_df['network_type'] == net_type]
        if net_data.empty: continue

        sankey_data = net_data.groupby(['blocker_strategy', 'adversary_strategy']).size().reset_index(name='count')
        all_strats = np.unique(list(sankey_data['blocker_strategy'].unique()) + list(sankey_data['adversary_strategy'].unique()))

        if net_type in EMPIRICAL_DATASETS:
            palette = ['#FF6B6B', '#4ECDC4', '#FFD166', '#06D6A0', '#118AB2', '#EF476F']
            colors = {s: palette[i % len(palette)] for i, s in enumerate(all_strats)}
        else:
            colors = {s: STRATEGY_COLOR_MAP.get(s, '#333333') for s in all_strats}

        labels = [f"{s} (Blocker)" for s in all_strats] + [f"{s} (Adversary)" for s in all_strats]
        node_colors = [colors[s] for s in all_strats] * 2
        node_map = {name: i for i, name in enumerate(labels)}

        source, target, value, link_colors = [], [], [], []
        for _, row in sankey_data.iterrows():
            idx = node_map[f"{row['blocker_strategy']} (Blocker)"]
            source.append(idx)
            target.append(node_map[f"{row['adversary_strategy']} (Adversary)"])
            value.append(row['count'])
            link_colors.append(hex_to_rgba(node_colors[idx], 0.4))

        fig = go.Figure(data=[go.Sankey(
            node=dict(pad=25, thickness=20, line=dict(color="black", width=0.5), label=labels, color=node_colors),
            link=dict(source=source, target=target, value=value, color=link_colors)
        )])

        fig.update_layout(title="")

        safe_name = net_type.replace('.', '_').replace('/', '_')
        fig.write_html(f'{OUTPUT_FOLDER}/fig_8_sankey_{safe_name}.html')
        if KALEIDO_INSTALLED:
             try: fig.write_image(f'{OUTPUT_FOLDER}/fig_8_sankey_{safe_name}.png', width=1200, height=800, scale=2)
             except: pass

# ============================================================================
# 9. FACETED PAYOFF HEATMAPS
# ============================================================================

def plot_fig9_faceted_payoff_heatmaps(combined_strategy_df, all_blocker_strategies, all_adversary_strategies):
    if combined_strategy_df.empty or 'norm_influence' not in combined_strategy_df.columns: return
    print(f"  Generating Figure 9: Faceted Payoff Heatmaps")

    empirical_data = combined_strategy_df[combined_strategy_df['network_type'].isin(EMPIRICAL_DATASETS)]
    if empirical_data.empty: return

    group_data = empirical_data.groupby(['network_type', 'legend_label', 'blocker_strategy', 'adversary_strategy'])['norm_influence'].mean()
    if group_data.empty: return
    plot_data = group_data.reset_index()

    cmap = plt.get_cmap('plasma'); cmap.set_bad(color=NAN_COLOR)

    def draw_heatmap(*args, **kwargs):
        data = kwargs.pop('data')
        if data.empty: return
        heatmap_data = data.pivot(index='blocker_strategy', columns='adversary_strategy', values='norm_influence')
        heatmap_data = heatmap_data.reindex(index=all_blocker_strategies, columns=all_adversary_strategies)
        sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap=cmap, cbar=False, **kwargs)

    g = sns.FacetGrid(plot_data, row='network_type', col='legend_label',
                      col_order=[cfg[2] for cfg in EXPERIMENTS_CONFIG],
                      row_order=EMPIRICAL_DATASETS, margin_titles=True, height=4)
    g.map_dataframe(draw_heatmap)
    g.set_titles(col_template="{col_name}", row_template="{row_name} (Empirical)")
    for ax in g.axes.flat:
        for label in ax.get_xticklabels(): label.set_rotation(90)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_FOLDER}/fig_9_faceted_payoff_heatmaps_empirical.png', bbox_inches='tight')
    plt.close()

# ============================================================================
# STATISTICAL TESTS
# ============================================================================

def statistical_tests(combined_df):
    if combined_df.empty: return
    print("\n" + "="*80 + "\nSTATISTICAL TESTS\n" + "="*80)

    print("\nHYPOTHESIS A: Independent vs. Minimax Blocker Performance")
    group_indep = combined_df[combined_df['blocker_type'] == 'independent']['norm_influence'].dropna()
    group_minimax = combined_df[combined_df['blocker_type'] == 'minimax']['norm_influence'].dropna()
    if not group_indep.empty and not group_minimax.empty:
        u_stat, p_value = mannwhitneyu(group_indep, group_minimax, alternative='two-sided')
        print(f"  Mann-Whitney U: U={u_stat:.2f}, p={p_value:.6f}")
        print(f"  Means: Indep={group_indep.mean():.4f}, Minimax={group_minimax.mean():.4f}")

    print("\nHYPOTHESIS B: Network Type Effects (including real-world)")
    if 'network_type' in combined_df.columns:
        network_types = combined_df['network_type'].dropna().unique()
        groups = [combined_df[combined_df['network_type'] == nt]['norm_influence'].dropna() for nt in network_types]
        if len(groups) > 1:
            h_stat, p_value = kruskal(*groups)
            print(f"  Kruskal-Wallis: H={h_stat:.4f}, p={p_value:.6f}")
            print(f"  Network types tested: {', '.join(network_types)}")

    print("\nHYPOTHESIS C: Synthetic vs Empirical Networks")
    if 'network_type' in combined_df.columns:
        synthetic_data = combined_df[~combined_df['network_type'].isin(EMPIRICAL_DATASETS)]['norm_influence'].dropna()
        real_world_data = combined_df[combined_df['network_type'].isin(EMPIRICAL_DATASETS)]['norm_influence'].dropna()
        if not synthetic_data.empty and not real_world_data.empty:
            u_stat_rw, p_value_rw = mannwhitneyu(synthetic_data, real_world_data, alternative='two-sided')
            print(f"  Mann-Whitney U (Synthetic vs Empirical): U={u_stat_rw:.2f}, p={p_value_rw:.6f}")
            print(f"  Means: Synthetic={synthetic_data.mean():.4f}, Empirical={real_world_data.mean():.4f}")

    print("\nHYPOTHESIS D: Network Properties vs. Influence (Spearman)")
    potential_props = ['avg_clustering', 'degree_entropy', 'structural_entropy']
    available_props = [p for p in potential_props if p in combined_df.columns]
    if available_props:
        agg_ops = {prop: 'mean' for prop in available_props}
        agg_ops['norm_influence'] = 'mean'
        agg_df = combined_df.groupby('network').agg(agg_ops).dropna()
        if not agg_df.empty:
            for prop in available_props:
                corr, p_value = spearmanr(agg_df[prop], agg_df['norm_influence'])
                print(f"  - {prop}: Rho={corr:.4f}, p={p_value:.6f}")

# ============================================================================
# MAIN
# ============================================================================

def analyze_overall_results(experiments):
    all_results = []
    for exp_id, data in experiments.items():
        if 'results' in data and not data['results'].empty:
            df = data['results'].copy()
            df['experiment'] = EXPERIMENT_SHORT_NAMES.get(exp_id, exp_id)
            df['legend_label'] = EXPERIMENT_LEGEND_LABELS.get(exp_id, exp_id)
            df['blocker_type'] = EXPERIMENT_AGENT_TYPES[exp_id]['blocker_type']
            df['adversary_type'] = EXPERIMENT_AGENT_TYPES[exp_id]['adversary_type']
            all_results.append(df)

    if not all_results: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    combined_df = pd.concat(all_results, ignore_index=True)
    combined_df['network_type'] = combined_df['network'].apply(extract_network_type)
    combined_df['phase'] = pd.Categorical(combined_df['phase'], categories=['train', 'test'], ordered=True)

    experiment_labels = [cfg[2] for cfg in EXPERIMENTS_CONFIG]

    if 'network_type' in combined_df.columns:
        unique_types = sorted(combined_df['network_type'].dropna().unique())
        markers_list = ['o', '^', 'h', 's', 'P', '*', 'D', 'v', '<', '>', 'p', 'H']
        marker_map = {nt: markers_list[i % len(markers_list)] for i, nt in enumerate(unique_types)}
    else: marker_map = {}

    df_train = combined_df[combined_df['phase'] == 'train']
    df_test = combined_df[combined_df['phase'] == 'test']

    plot_fig1a_influence_boxplot(df_train, 'TRAIN')
    plot_fig1a_influence_boxplot(df_test, 'TEST')
    plot_fig1a_influence_boxplot(combined_df, 'COMBINED')

    plot_fig1b_generalization_barplot(df_train, 'TRAIN')
    plot_fig1b_generalization_barplot(df_test, 'TEST')
    plot_fig1b_generalization_barplot(combined_df, 'COMBINED')

    prop_list = [('avg_clustering', 'Avg Clustering'), ('degree_entropy', 'Degree Entropy'), ('structural_entropy', 'Structural Entropy')]
    for prop, label in prop_list:
        plot_fig2_regplot(combined_df, prop, label, 'COMBINED', experiment_labels, marker_map)

    all_strategy_dfs = []
    for exp_id, data in experiments.items():
        if 'strategy' in data and not data['strategy'].empty:
            df = data['strategy'].copy()
            df['experiment'] = EXPERIMENT_SHORT_NAMES.get(exp_id, exp_id)
            df['legend_label'] = EXPERIMENT_LEGEND_LABELS.get(exp_id, exp_id)
            df['blocker_type'] = EXPERIMENT_AGENT_TYPES[exp_id]['blocker_type']
            df['adversary_type'] = EXPERIMENT_AGENT_TYPES[exp_id]['adversary_type']
            df['blocker_strategy'] = df['blocker_strategy_id'].apply(map_strategy_id_to_name)
            df['adversary_strategy'] = df['adversary_strategy_id'].apply(map_strategy_id_to_name)
            df['network_type'] = df['network'].apply(extract_network_type)

            if 'results' in data and not data['results'].empty:
                 res = data['results'][['network', 'game_num', 'norm_influence']]
                 df = pd.merge(df, res, on=['network', 'game_num'], how='left')
            else:
                 df['norm_influence'] = np.nan

            all_strategy_dfs.append(df)

    all_trace_dfs = []
    for exp_id, data in experiments.items():
        if 'trace' in data and not data['trace'].empty:
            df = data['trace'].copy()
            df['legend_label'] = EXPERIMENT_LEGEND_LABELS.get(exp_id, exp_id)
            df['network_type'] = df['network'].apply(extract_network_type)
            all_trace_dfs.append(df)

    combined_strategy = pd.concat(all_strategy_dfs, ignore_index=True) if all_strategy_dfs else pd.DataFrame()
    combined_trace = pd.concat(all_trace_dfs, ignore_index=True) if all_trace_dfs else pd.DataFrame()

    return combined_df, combined_strategy, combined_trace

def main():
    print(f"MULTI-EXPERIMENT ANALYSIS WITH EMPIRICAL DATASETS\n")
    if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)
    experiments = load_all_experiments()
    if not experiments: return

    combined_df, combined_strategy_df, combined_trace_df = analyze_overall_results(experiments)

    if not combined_trace_df.empty:
        plot_fig3_consolidated_traces(combined_trace_df)
        for emp_dataset in EMPIRICAL_DATASETS:
            emp_data = combined_trace_df[combined_trace_df['network_type'] == emp_dataset]
            if not emp_data.empty:
                for net in emp_data['network'].unique():
                    plot_fig3_single_network_trace(emp_data[emp_data['network'] == net], net)

    if not combined_strategy_df.empty:
        if 'network_type' in combined_strategy_df.columns:
            for net_type in sorted(combined_strategy_df['network_type'].dropna().unique()):
                # Pass BOTH combined_strategy_df (filtered) AND combined_df (full results)
                plot_fig4_strategy_evolution(
                    combined_strategy_df[combined_strategy_df['network_type'] == net_type],
                    combined_df, # Pass full results for correct stat calculation
                    f"Strategy Evolution for {net_type}",
                    f'{OUTPUT_FOLDER}/fig_4_consolidated_{net_type.replace(".", "_")}_strategy.png'
                )

        plot_fig5_strategy_frequency_heatmap(combined_strategy_df)
        plot_fig6_payoff_matrix(combined_strategy_df)

        all_b = sorted(combined_strategy_df['blocker_strategy'].dropna().unique())
        all_a = sorted(combined_strategy_df['adversary_strategy'].dropna().unique())

        plot_fig7_faceted_frequency_heatmaps(combined_strategy_df, all_b, all_a)

        plot_fig8_network_type_sankeys(combined_strategy_df)
        plot_fig9_faceted_payoff_heatmaps(combined_strategy_df, all_b, all_a)

    statistical_tests(combined_df)
    print(f"\nANALYSIS COMPLETE. Figures saved to '{OUTPUT_FOLDER}'.")

if __name__ == "__main__":
    main()

MULTI-EXPERIMENT ANALYSIS WITH EMPIRICAL DATASETS

Loaded experiment data for: e1 (as Indep_vs_Indep)
Loaded experiment data for: e2 (as Minimax_vs_Indep)
Loaded experiment data for: e3 (as Minimax_vs_Minimax)
Loaded experiment data for: e4 (as Indep_vs_Minimax)
  Generating Figure 1a: Influence Boxplot (TRAIN)
  Generating Figure 1a: Influence Boxplot (TEST)
  Generating Figure 1a: Influence Boxplot (COMBINED)
  Generating Figure 1b: Generalization Barplot (TRAIN)
  Generating Figure 1b: Generalization Barplot (TEST)
  Generating Figure 1b: Generalization Barplot (COMBINED)
  Generating Figure 2: Regression Plot for avg_clustering (COMBINED)
  Generating Figure 2: Regression Plot for degree_entropy (COMBINED)
  Generating Figure 2: Regression Plot for structural_entropy (COMBINED)
  Generating Figure 3 (Consolidated) for Network Type: BA
  Generating Figure 3 (Consolidated) for Network Type: EMAIL
  Generating Figure 3 (Consolidated) for Network Type: ER
  Generating Figure 3 (Consoli

In [ ]:
import shutil
import os
from google.colab import files

# Define the folder to zip and the name for the zip file
folder_to_zip = 'analysis_figures'
zip_filename = f'{folder_to_zip}.zip'

# Create the zip file
if os.path.exists(folder_to_zip):
    print(f"Zipping folder: {folder_to_zip}")
    shutil.make_archive(folder_to_zip, 'zip', folder_to_zip)
    print(f"Created zip file: {zip_filename}")

    # Download the zip file
    if os.path.exists(zip_filename):
        print(f"Downloading: {zip_filename}")
        files.download(zip_filename)
    else:
        print(f"Error: Zip file '{zip_filename}' was not created.")
else:
    print(f"Error: Folder '{folder_to_zip}' not found.")

Zipping folder: analysis_figures
Created zip file: analysis_figures.zip
Downloading: analysis_figures.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>